# Transfer Learning - Pre-trained Weight Loading

This notebook demonstrates loading pre-trained weights for sequence and structure encoders.

In [ ]:
# Setup
import torch
from pathlib import Path
import sys
sys.path.append('..')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Import model
from src.models.biophystcr import BioPhysTCR, BioPhysTCRConfig

# Create model with correct dimensions
config = BioPhysTCRConfig(
    esm2_dim=1280,
    sequence_hidden_dim=256,
    sequence_num_layers=2,
    saprot_dim=446,
    structure_hidden_dim=256,
    structure_num_gnn_layers=3,
    physchem_dim=8,
    physchem_hidden_dim=64,
    fusion_dim=256,
    projection_dim=128,
    dropout=0.2
)

model = BioPhysTCR(config).cuda()
print(f"✅ Model created: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

In [ ]:
# Weight Loading Functions

def load_sequence_weights(target_module, weights_dict, prefix):
    """Load pre-trained sequence encoder weights."""
    target_state = target_module.state_dict()
    updates = {}
    loaded = 0
    
    mappings = {
        'input_projection.weight': f'{prefix}.src_emb.weight',
        'input_projection.bias': f'{prefix}.src_emb.bias',
        'pos_encoding.pe': f'{prefix}.pos_emb.pe',
    }
    
    for i in range(2):
        mappings[f'transformer.layers.{i}.linear1.weight'] = f'{prefix}.layers.{i}.pos_ffn.fc.0.weight'
        mappings[f'transformer.layers.{i}.linear1.bias'] = f'{prefix}.layers.{i}.pos_ffn.fc.0.bias'
        mappings[f'transformer.layers.{i}.linear2.weight'] = f'{prefix}.layers.{i}.pos_ffn.fc.3.weight'
        mappings[f'transformer.layers.{i}.linear2.bias'] = f'{prefix}.layers.{i}.pos_ffn.fc.3.bias'
    
    for our_key, source_key in mappings.items():
        if source_key not in weights_dict:
            continue
            
        tensor = weights_dict[source_key]
        our_shape = target_state[our_key].shape
        
        if 'pos_encoding.pe' in our_key:
            if tensor.shape != our_shape:
                sliced = tensor[:our_shape[1], :, :].permute(1, 0, 2)
                if sliced.shape == our_shape:
                    updates[our_key] = sliced
                    loaded += 1
            continue
        
        if tensor.shape == our_shape:
            updates[our_key] = tensor
            loaded += 1
    
    target_module.load_state_dict(updates, strict=False)
    return loaded

def load_structure_weights(target_module, weights_dict):
    """Load pre-trained structure encoder weights."""
    target_state = target_module.state_dict()
    updates = {}
    loaded = 0
    
    if 'embedding.res_fc.weight' in weights_dict:
        updates['embedding.fc.weight'] = weights_dict['embedding.res_fc.weight']
        updates['embedding.fc.bias'] = weights_dict['embedding.res_fc.bias']
        loaded += 1
    
    for i in range(3):
        src_prefix = f'res_module.gnn_layer.conv{i+1}'
        tgt_prefix = f'gnn.convs.{i}'
        
        for suffix in ['.lin_l.weight', '.lin_l.bias', '.lin_r.weight']:
            src_key = src_prefix + suffix
            if src_key in weights_dict:
                updates[tgt_prefix + suffix] = weights_dict[src_key]
                loaded += 1

    target_module.load_state_dict(updates, strict=False)
    return loaded

print("✅ Loading functions defined")

In [ ]:
# Load Sequence Encoder Weights
print("\n" + "="*50)
print("LOADING SEQUENCE ENCODER WEIGHTS")
print("="*50)

seq_path = Path('../pretrained_weights/sequence_encoder.pth')
if seq_path.exists():
    seq_data = torch.load(seq_path, map_location='cpu', weights_only=True)
    if 'model_state_dict' in seq_data:
        seq_data = seq_data['model_state_dict']
    
    tcr_loaded = load_sequence_weights(model.tcr_sequence_encoder.encoder, seq_data, 'b_encoder')
    pmhc_loaded = load_sequence_weights(model.pmhc_sequence_encoder.epitope_encoder, seq_data, 'p_encoder')
    
    print(f"\n✅ Sequence: {tcr_loaded} tensors (TCR), {pmhc_loaded} tensors (pMHC)")
else:
    print("❌ Sequence weights not found!")

In [ ]:
# Load Structure Encoder Weights
print("\n" + "="*50)
print("LOADING STRUCTURE ENCODER WEIGHTS")
print("="*50)

struct_path = Path('../pretrained_weights/structure_encoder.pt')
if struct_path.exists():
    struct_data = torch.load(struct_path, map_location='cpu', weights_only=True)
    
    tcr_loaded = load_structure_weights(model.tcr_structure_encoder, struct_data)
    pmhc_loaded = load_structure_weights(model.pmhc_structure_encoder, struct_data)
    
    print(f"\n✅ Structure: {tcr_loaded} tensors (TCR), {pmhc_loaded} tensors (pMHC)")
else:
    print("❌ Structure weights not found!")

In [ ]:
# Summary
print("\n" + "="*50)
print("TRANSFER LEARNING SETUP COMPLETE")
print("="*50)
print("\n✅ Dual transfer learning initialized:")
print("   • Pre-trained sequence encoders (ESM2 dim=1280)")
print("   • Pre-trained structure encoders (SaProt dim=446)")
print("\nModel is ready for training!")